In [7]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

#spilloverを考慮したSCM感度分析
"""
この感度分析の目的は、「ドナー群（比較対象）に介入の波及効果があるときに、SCMの推定精度がどれほど悪化するか」を確認すること。
SCMの計算結果は「処置群の実績値」と「合成コントロール（反実仮想）の予測値」の差分（Gap）の平均値。
そのため商圏が重複する対照群への流出によって処置群の数値が減っているかどうかに関わらず、
「ドナー側の来訪者が増えてしまうこと自体が分析の敵（処置効果の過小評価につながってしまう）」であるため、
まずは介入によるドナー側の増加（汚染）のみをシンプルにシミュレーションする
"""
input_path = Path("/content/drive/MyDrive/因果推論/h3_mesh_panel.csv")
output_dir = input_path.parent

intervention_date = pd.Timestamp("2025-01-01")
#介入の影響が処置を受けていない周辺地域（ドナーユニット）にどれくらいの強さで波及すると仮定するかを示すパラメータ
spillover_strengths = [0.00, 0.25, 0.50, 1.00]
#処置エリアとの商圏重複率（overlap_score）がこの値以上であるドナーユニットを、分析から除外するためのしきい値
#スピルオーバーの影響を受けている可能性が高いドナー（対照群）を、分析から除外するためのしきい値
exclusion_thresholds = [1.01, 0.70, 0.60, 0.50, 0.30]

#このCSVには実測の距離・隣接関係・商圏重複率がないため、教材用にdonor_groupとJaccard係数を設定する。駅名や実地域は表さない。
teaching_groups = {
    "donor_group_1": {"h3_ids": [f"h3_mesh_{i:02d}" for i in range(7, 13)], "overlap_score": 0.75},
    "donor_group_2": {"h3_ids": [f"h3_mesh_{i:02d}" for i in range(13, 19)], "overlap_score": 0.45},
    "donor_group_3": {"h3_ids": [f"h3_mesh_{i:02d}" for i in range(19, 25)], "overlap_score": 0.25},
    "donor_group_4": {"h3_ids": [f"h3_mesh_{i:02d}" for i in range(25, 31)], "overlap_score": 0.20},
    "donor_group_5": {"h3_ids": [f"h3_mesh_{i:02d}" for i in range(31, 37)], "overlap_score": 0.65},
}
display(teaching_groups)

def rmspe(values):
    values = np.asarray(values, dtype=float)
    return float(np.sqrt(np.mean(values ** 2)))

#spillover_strengths と exclusion_thresholds のすべての組み合わせ、つまり各シナリオに対して
#合成コントロール法の計算を行い、その結果を返すfit_scm()関数を定義
def fit_scm(panel, strength, threshold, true_effect):
    work = panel.copy()
    #work["assumed_spillover"] :仮定された波及効果によるドナーユニットの来訪者数の「かさ上げ分」を人工的に作り出した値。
    work["assumed_spillover"] = np.where((work["treated"] == 0) & (work["date"] >= intervention_date),
        #商圏重複率 * 未処置の領域に波及して影響を仮定した効果 * 真の処置効果
        work["overlap_score"] * strength * true_effect,0.0)
    #work["visitors_scenario"] はドナー群に波及効果の増加分を踏まえた観測者数
    work["visitors_scenario"] = work["visitors"] + work["assumed_spillover"]
    #対照群のうち、商圏重複率が特定のしきい値以上であるドナーグループを特定し、それらを分析から除外するリストを作成する
    excluded_groups = sorted(
        work.loc[(work["treated"] == 0) & (work["overlap_score"] >= threshold),"donor_group"
        ].dropna().unique())
    #商圏重複率のしきい値を変えて、商圏重複率が大きいエリアを段階的に除外してみる。
    donor_part = work.loc[(work["treated"] == 0) & (work["overlap_score"] < threshold)]
    donor_ids = sorted(donor_part["h3_id"].unique())
    if not donor_ids:
        raise ValueError(f"threshold={threshold}ではドナーが0件です。")
    #処置系列を作成
    actual = work.loc[work["treated"] == 1].groupby("date")["visitors_scenario"].mean().sort_index()
    #商圏重複率の閾値を超えたメッシュ除外後のドナー系列を作成
    donor_wide = (
        donor_part.pivot(index="date", columns="h3_id", values="visitors_scenario")
        .sort_index().loc[:, donor_ids]
    )
    dates = actual.index.intersection(donor_wide.index)
    actual = actual.loc[dates]
    donor_wide = donor_wide.loc[dates]
    pre_mask = dates < intervention_date
    #pre_mask のブール値を反転させたもの（論理否定)
    post_mask = ~pre_mask
    y_pre = actual.loc[pre_mask].to_numpy(float)
    x_pre = donor_wide.loc[pre_mask].to_numpy(float)

    def objective(weights):
        residual = y_pre - x_pre @ weights
        return np.mean(residual ** 2) / np.var(y_pre)

    donor_count = len(donor_ids)
    result = minimize(
        objective,
        np.full(donor_count, 1.0 / donor_count),
        method="SLSQP",
        bounds=[(0.0, 1.0)] * donor_count,
        constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
        options={"ftol": 1e-12, "maxiter": 5000, "disp": False}
    )
    if not result.success:
        raise RuntimeError(f"SCM最適化失敗: {result.message}")

    weights = result.x
    synthetic = donor_wide.to_numpy(float) @ weights
    series = pd.DataFrame({
        "date": dates,
        "actual_treated_area": actual.to_numpy(float),
        "synthetic_treated_area": synthetic
    })
    series["gap"] = series["actual_treated_area"] - series["synthetic_treated_area"]
    series["period"] = np.where(series["date"] < intervention_date, "pre", "post")
    pre_gap = series.loc[series["period"] == "pre", "gap"]
    post_gap = series.loc[series["period"] == "post", "gap"]
    pre_rmspe = rmspe(pre_gap)
    post_rmspe = rmspe(post_gap)
    scenario_id = f"strength_{strength:.2f}__threshold_{threshold:.2f}"
    series["scenario_id"] = scenario_id
    display(series)

    summary = {
        "scenario_id": scenario_id,
        "spillover_strength": strength,
        "exclusion_threshold": threshold,
        "excluded_groups": "|".join(excluded_groups) if excluded_groups else "none",
        "donor_mesh_count": donor_count,
        "pre_rmspe": pre_rmspe,
        "post_rmspe": post_rmspe,
        "post_pre_rmspe_ratio": post_rmspe / pre_rmspe,
        "mean_post_gap": float(post_gap.mean()),
        "true_direct_effect": true_effect,
        "estimation_error": float(post_gap.mean() - true_effect),
    }
    weights_df = pd.DataFrame({
        "scenario_id": scenario_id,
        "donor_h3_id": donor_ids,
        "weight": weights
    })
    display(weights_df)
    return summary, series, weights_df

#データの読み込み・データの確認
df = pd.read_csv(input_path, parse_dates=["date"])
#必要列が読み込んだデータフレームに揃っているかを確認
required = {"date", "h3_id", "visitors", "treated", "post", "treatment_effect_true"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"必要列がありません: {sorted(missing)}")

#データ型を指定
df["h3_id"] = df["h3_id"].astype(str)
df["visitors"] = pd.to_numeric(df["visitors"], errors="raise")
df["treated"] = pd.to_numeric(df["treated"], errors="raise").astype(int)
df["post"] = pd.to_numeric(df["post"], errors="raise").astype(int)
df["treatment_effect_true"] = pd.to_numeric(df["treatment_effect_true"], errors="raise")

#重複している行がないか確認
if df.duplicated(["date", "h3_id"]).any():
    raise ValueError("date × h3_idに重複があります。")
#欠損値がないか確認
if df.isna().any().any():
    raise ValueError("欠損値があります。")
#適切にデータが揃っているか確認（※2024年はうるう年であり、366日存在）
date_counts = df.groupby("h3_id")["date"].nunique()
display(date_counts)
#date_counts Seriesに含まれるユニークな値の数を数えています。
#date_counts Seriesは、各h3_idが持っているユニークな日付の数を値として持っています
if date_counts.nunique() != 1 or date_counts.iloc[0] != df["date"].nunique():
    raise ValueError("バランスパネルではありません。")
#post列が介入日に対し正しく反映されているかを確認
expected_post = (df["date"] >= intervention_date).astype(int)
display(expected_post)
if not np.array_equal(df["post"].to_numpy(), expected_post.to_numpy()):
    raise ValueError("post列と介入日の定義が一致しません。")
#処置群を表すtreatedが1になっているか（0を含んでいないか）確認
if df.groupby("h3_id")["treated"].nunique().max() != 1:
    raise ValueError("同じh3_id内でtreatedが変化しています。")
#処置群のメッシュの集合を作成
treated_ids = set(df.loc[df["treated"] == 1, "h3_id"].unique())
#対照群のメッシュの集合を作成
control_ids = set(df.loc[df["treated"] == 0, "h3_id"].unique())
#teaching_groupsでドナー群として定義したh3_id のユニークな集合を作成
declared_ids = {h3_id for group in teaching_groups.values() for h3_id in group["h3_ids"]}
display(declared_ids)
if control_ids != declared_ids:
    raise ValueError(
        "CSVの対照h3_idと教材用グループ定義が一致しません。\n"
        f"CSVのみに存在: {sorted(control_ids - declared_ids)}\n"
        f"定義のみに存在: {sorted(declared_ids - control_ids)}"
    )
#処置群のメッシュがドナー群のメッシュに混入していないか確認
if treated_ids & declared_ids:
    raise ValueError("処置メッシュが教材用ドナー群へ混入しています。")

#メッシュに関するメタデータを結合
metadata_rows = []
for group_name, setting in teaching_groups.items():
    for h3_id in setting["h3_ids"]:
        metadata_rows.append({
            "h3_id": h3_id,
            "donor_group": group_name,
            "overlap_score": setting["overlap_score"],
            "score_status": "artificial_for_teaching_not_observed",
        })
metadata = pd.DataFrame(metadata_rows)
df = df.merge(metadata, on="h3_id", how="left", validate="many_to_one")
#結合によって生じた欠損を補完
df.loc[df["treated"] == 1, "donor_group"] = "treated_area"
df.loc[df["treated"] == 1, "overlap_score"] = 0.0
df.loc[df["treated"] == 1, "score_status"] = "not_applicable"
#true_effect の値が有効であることを保証し、後続の計算に進む前のエラーハンドリング
true_effect = float(
    df.loc[(df["treated"] == 1) & (df["post"] == 1), "treatment_effect_true"].mean()
)
if not np.isfinite(true_effect):
    raise ValueError("施策後の真の直接効果を取得できません。")

#全20シナリオを推定
summaries, series_list, weights_list = [], [], []
for strength in spillover_strengths:
    for threshold in exclusion_thresholds:
        summary, series, weights = fit_scm(df, strength, threshold, true_effect)
        summaries.append(summary)
        series_list.append(series)
        weights_list.append(weights)

summary_df = pd.DataFrame(summaries)
print(f"summary_df \n")
display(summary_df)
series_df = pd.concat(series_list, ignore_index=True)
print(f"series_df series_df : 20シナリオ * 731日（閏年含む）/シナリオ = 14620行　\n")
display(series_df)
weights_df = pd.concat(weights_list, ignore_index=True)
print(f"weights_df : 総行数は、20個の各シナリオで選ばれたドナーメッシュの数をすべて合計したもの\n")
display(weights_df)

#出力をCSVで保存
metadata.to_csv(output_dir / "teaching_spillover_metadata.csv", index=False)
summary_df.to_csv(output_dir / "spillover_sensitivity_summary.csv", index=False)
series_df.to_csv(output_dir / "spillover_sensitivity_series.csv", index=False)
weights_df.to_csv(output_dir / "spillover_sensitivity_weights.csv", index=False)

audit_df = pd.DataFrame([{
    "input_path": str(input_path),
    "date_min": df["date"].min(),
    "date_max": df["date"].max(),
    "date_count": df["date"].nunique(),
    "row_count": len(df),
    "mesh_count": df["h3_id"].nunique(),
    "treated_mesh_count": df.loc[df["treated"] == 1, "h3_id"].nunique(),
    "donor_mesh_count": df.loc[df["treated"] == 0, "h3_id"].nunique(),
    "duplicate_date_h3_count": int(df.duplicated(["date", "h3_id"]).sum()),
    "missing_value_count": int(df.isna().sum().sum()),
    "true_direct_effect": true_effect,
    "spillover_formula": "strength * artificial_overlap_score * true_effect"
}])
display(audit_df)
audit_df.to_csv(output_dir / "data_audit.csv", index=False)



{'donor_group_1': {'h3_ids': ['h3_mesh_07',
   'h3_mesh_08',
   'h3_mesh_09',
   'h3_mesh_10',
   'h3_mesh_11',
   'h3_mesh_12'],
  'overlap_score': 0.75},
 'donor_group_2': {'h3_ids': ['h3_mesh_13',
   'h3_mesh_14',
   'h3_mesh_15',
   'h3_mesh_16',
   'h3_mesh_17',
   'h3_mesh_18'],
  'overlap_score': 0.45},
 'donor_group_3': {'h3_ids': ['h3_mesh_19',
   'h3_mesh_20',
   'h3_mesh_21',
   'h3_mesh_22',
   'h3_mesh_23',
   'h3_mesh_24'],
  'overlap_score': 0.25},
 'donor_group_4': {'h3_ids': ['h3_mesh_25',
   'h3_mesh_26',
   'h3_mesh_27',
   'h3_mesh_28',
   'h3_mesh_29',
   'h3_mesh_30'],
  'overlap_score': 0.2},
 'donor_group_5': {'h3_ids': ['h3_mesh_31',
   'h3_mesh_32',
   'h3_mesh_33',
   'h3_mesh_34',
   'h3_mesh_35',
   'h3_mesh_36'],
  'overlap_score': 0.65}}

,date
h3_id,
h3_mesh_01,731
h3_mesh_02,731
h3_mesh_03,731
h3_mesh_04,731
h3_mesh_05,731
h3_mesh_06,731
h3_mesh_07,731
h3_mesh_08,731
h3_mesh_09,731


,date
0,0
1,0
2,0
3,0
4,0
...,...
26311,1
26312,1
26313,1
26314,1


{'h3_mesh_07',
 'h3_mesh_08',
 'h3_mesh_09',
 'h3_mesh_10',
 'h3_mesh_11',
 'h3_mesh_12',
 'h3_mesh_13',
 'h3_mesh_14',
 'h3_mesh_15',
 'h3_mesh_16',
 'h3_mesh_17',
 'h3_mesh_18',
 'h3_mesh_19',
 'h3_mesh_20',
 'h3_mesh_21',
 'h3_mesh_22',
 'h3_mesh_23',
 'h3_mesh_24',
 'h3_mesh_25',
 'h3_mesh_26',
 'h3_mesh_27',
 'h3_mesh_28',
 'h3_mesh_29',
 'h3_mesh_30',
 'h3_mesh_31',
 'h3_mesh_32',
 'h3_mesh_33',
 'h3_mesh_34',
 'h3_mesh_35',
 'h3_mesh_36'}

,date,actual_treated_area,synthetic_treated_area,gap,period,scenario_id
0,2024-01-01,1269.032359,1274.327739,-5.295380,pre,strength_0.00__threshold_1.01
1,2024-01-02,1289.206883,1286.982157,2.224727,pre,strength_0.00__threshold_1.01
2,2024-01-03,1305.904895,1298.944127,6.960768,pre,strength_0.00__threshold_1.01
3,2024-01-04,1316.002988,1310.357281,5.645707,pre,strength_0.00__threshold_1.01
4,2024-01-05,1338.300240,1323.250608,15.049632,pre,strength_0.00__threshold_1.01
...,...,...,...,...,...,...
726,2025-12-27,1621.734952,1398.428329,223.306623,post,strength_0.00__threshold_1.01
727,2025-12-28,1613.795981,1385.643640,228.152341,post,strength_0.00__threshold_1.01
728,2025-12-29,1541.604173,1326.986701,214.617472,post,strength_0.00__threshold_1.01
729,2025-12-30,1560.426800,1341.274830,219.151971,post,strength_0.00__threshold_1.01


...

series_df series_df : 20シナリオ * 731日（閏年含む）/シナリオ = 14620行　



,date,actual_treated_area,synthetic_treated_area,gap,period,scenario_id
0,2024-01-01,1269.032359,1274.327739,-5.295380,pre,strength_0.00__threshold_1.01
1,2024-01-02,1289.206883,1286.982157,2.224727,pre,strength_0.00__threshold_1.01
2,2024-01-03,1305.904895,1298.944127,6.960768,pre,strength_0.00__threshold_1.01
3,2024-01-04,1316.002988,1310.357281,5.645707,pre,strength_0.00__threshold_1.01
4,2024-01-05,1338.300240,1323.250608,15.049632,pre,strength_0.00__threshold_1.01
...,...,...,...,...,...,...
14615,2025-12-27,1621.734952,1454.467423,167.267529,post,strength_1.00__threshold_0.30
14616,2025-12-28,1613.795981,1436.906998,176.888983,post,strength_1.00__threshold_0.30
14617,2025-12-29,1541.604173,1370.463595,171.140578,post,strength_1.00__threshold_0.30
14618,2025-12-30,1560.426800,1387.356935,173.069865,post,strength_1.00__threshold_0.30


weights_df : 総行数は、20個の各シナリオで選ばれたドナーメッシュの数をすべて合計したもの



,scenario_id,donor_h3_id,weight
0,strength_0.00__threshold_1.01,h3_mesh_07,3.259500e-02
1,strength_0.00__threshold_1.01,h3_mesh_08,8.682807e-02
2,strength_0.00__threshold_1.01,h3_mesh_09,8.343769e-02
3,strength_0.00__threshold_1.01,h3_mesh_10,1.143291e-01
4,strength_0.00__threshold_1.01,h3_mesh_11,1.219556e-21
...,...,...,...
403,strength_1.00__threshold_0.30,h3_mesh_26,4.343876e-02
404,strength_1.00__threshold_0.30,h3_mesh_27,2.388176e-01
405,strength_1.00__threshold_0.30,h3_mesh_28,0.000000e+00
406,strength_1.00__threshold_0.30,h3_mesh_29,2.579806e-01


,input_path,date_min,date_max,date_count,row_count,mesh_count,treated_mesh_count,donor_mesh_count,duplicate_date_h3_count,missing_value_count,true_direct_effect,spillover_formula
0,/content/drive/MyDrive/因果推論/h3_mesh_panel.csv,2024-01-01,2025-12-31,731,26316,36,6,30,0,0,220.0,strength * artificial_overlap_score * true_effect
